In [13]:
import torch
import torch.nn as nn

In [14]:
# Cross Correlation operation 
# Out = X, K
# X : feature input / channel input
# K : Convolution Kernel

def corr2d(X: torch.Tensor, K: torch.Tensor) -> torch.Tensor:
    h, w = K.shape
    Y = torch.zeros(
        (X.shape[0] - h + 1, X.shape[1] - w + 1)
    )
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (
                X[i: i + h, j: j + w] * K
            ).sum()
    return Y

def corr2d_multi_channel(X, K):
    # Iterate through the 0th dimension (channel) of K first, then add them up
    return sum(corr2d(x, k) for x, k in zip(X, K))

In [15]:
X = torch.tensor([[[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]],
               [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]])
K = torch.tensor([[[0.0, 1.0], [2.0, 3.0]], [[1.0, 2.0], [3.0, 4.0]]])

corr2d_multi_channel(X, K)

tensor([[ 56.,  72.],
        [104., 120.]])

In [16]:
def corr2d_multi_channel_in_and_out(X, K):
    return torch.stack(
        [corr2d_multi_channel(X, k) for k in K],
        0
    )

In [17]:
K = torch.stack(
    (K, K + 1, K + 2), 0
)
K.shape

torch.Size([3, 2, 2, 2])

In [18]:
corr2d_multi_channel_in_and_out(X, K)

tensor([[[ 56.,  72.],
         [104., 120.]],

        [[ 76., 100.],
         [148., 172.]],

        [[ 96., 128.],
         [192., 224.]]])

In [19]:
def corr2d_multi_in_out_1x1(X, K):
    c_i, h, w = X.shape
    c_o = K.shape[0]
    X = X.reshape((c_i, h * w))
    K = K.reshape((c_o, c_i))
    # Matrix multiplication in the fully connected layer
    Y = torch.matmul(K, X)
    return Y.reshape((c_o, h, w))

In [21]:
X = torch.normal(0, 1, (3, 3, 3))
K = torch.normal(0, 1, (2, 3, 1, 1))
Y1 = corr2d_multi_in_out_1x1(X, K)
Y2 = corr2d_multi_channel_in_and_out(X, K)
assert float(torch.abs(Y1 - Y2).sum()) < 1e-6
Y1

tensor([[[ 0.8467, -0.6089,  0.2333],
         [ 1.9032,  0.3834, -0.5467],
         [ 0.9978,  0.7231,  0.8925]],

        [[-0.9312,  0.4181,  0.7591],
         [-1.2244, -1.0616, -0.0195],
         [-2.1599, -0.7002, -0.6366]]])

In [22]:
Y2

tensor([[[ 0.8467, -0.6089,  0.2333],
         [ 1.9032,  0.3834, -0.5467],
         [ 0.9978,  0.7231,  0.8925]],

        [[-0.9312,  0.4181,  0.7591],
         [-1.2244, -1.0616, -0.0195],
         [-2.1599, -0.7002, -0.6366]]])